# Bayes Series | Chapter 4: Bayesian Linear Regression

> **Chapter Goal:** Move beyond point-estimate regression (OLS/MLE, Ridge) to full Bayesian linear regression — a framework that treats the model weights as random variables, gives a full posterior distribution over them, and produces **uncertainty-aware predictions**. Prove that Ridge regression IS MAP estimation, derive the closed-form posterior and posterior predictive distribution, and understand why Bayesian regression produces wider predictions in regions with few data points.

---

## 1. Standard Linear Regression as MLE

### **The Model**
Given data $\{(\vec{x}_i, y_i)\}_{i=1}^N$ with $\vec{x}_i \in \mathbb{R}^d$, $y_i \in \mathbb{R}$:
$$y_i = \vec{x}_i^T \vec{w} + \epsilon_i, \quad \epsilon_i \sim \mathcal{N}(0, \sigma^2)$$

This is a **probabilistic model** — $\epsilon$ is the observation noise.

### **MLE = Least Squares**
The likelihood for all data points:
$$P(\vec{y} | X, \vec{w}) = \prod_{i=1}^N \mathcal{N}(y_i; \vec{x}_i^T\vec{w}, \sigma^2) \propto \exp\left(-\frac{1}{2\sigma^2}\|\vec{y} - X\vec{w}\|^2\right)$$

Maximizing the log-likelihood (equivalently minimizing the negative log-likelihood):
$$\hat{\vec{w}}_{\text{MLE}} = \arg\min_{\vec{w}} \|\vec{y} - X\vec{w}\|^2 = (X^TX)^{-1}X^T\vec{y}$$

**MLE gives the ordinary least squares (OLS) solution.** No bias correction, no regularization — it will overfit on small datasets.

### **Limitation: No Uncertainty**
OLS gives a **point estimate** $\hat{\vec{w}}$ with no notion of how confident we are. Predictions $\hat{y}^* = \vec{x}^{*T}\hat{\vec{w}}$ are also point estimates with no uncertainty — they look equally confident everywhere, even far from training data.

---

## 2. MAP Estimation = Ridge Regression

### **Add a Gaussian Prior on Weights**
$$\vec{w} \sim \mathcal{N}(\vec{0}, \tau^2 I_d)$$

The prior encodes the belief that weights should be small (close to zero). The strength of this belief is $1/\tau^2$: small $\tau$ = strong belief weights are near zero.

### **MAP Derivation**
$$\hat{\vec{w}}_{\text{MAP}} = \arg\max_{\vec{w}} \underbrace{\log P(\vec{y}|X,\vec{w})}_{\text{log-likelihood}} + \underbrace{\log P(\vec{w})}_{\text{log-prior}}$$

$$= \arg\max_{\vec{w}} \left[-\frac{1}{2\sigma^2}\|\vec{y} - X\vec{w}\|^2 - \frac{1}{2\tau^2}\|\vec{w}\|^2\right]$$

Multiplying by $-2\sigma^2$ and defining $\lambda = \sigma^2/\tau^2$:

$$\boxed{\hat{\vec{w}}_{\text{MAP}} = \arg\min_{\vec{w}} \|\vec{y} - X\vec{w}\|^2 + \lambda\|\vec{w}\|^2 = (X^TX + \lambda I)^{-1} X^T \vec{y}}$$

**This is exactly Ridge regression!** MAP estimation with a Gaussian prior on weights ↔ L2 regularization with $\lambda = \sigma^2/\tau^2$.

| Ridge Hyperparameter | Bayesian Interpretation |
| :--- | :--- |
| $\lambda$ large | Strong prior ($\tau^2$ small) — weights strongly pulled toward 0 |
| $\lambda$ small | Weak prior ($\tau^2$ large) — MAP ≈ OLS |
| $\lambda = 0$ | Flat/uniform prior — MAP = MLE = OLS |

**But MAP is still a point estimate.** We haven't quantified uncertainty in $\vec{w}$. For that, we need the full posterior.

---

## 3. The Full Posterior Distribution over Weights

### **Setup**
Prior: $\vec{w} \sim \mathcal{N}(\vec{\mu}_0, \Sigma_0)$ (general Gaussian prior; simplify later with $\vec{\mu}_0 = \vec{0}$, $\Sigma_0 = \tau^2 I$)

Likelihood: $\vec{y} | X, \vec{w} \sim \mathcal{N}(X\vec{w}, \sigma^2 I_N)$

Since both are Gaussian and the likelihood is linear in $\vec{w}$, the posterior is also **Gaussian** (Gaussian-Gaussian conjugacy).

### **Posterior in Precision Form**
Define:
- **Prior precision matrix:** $\Lambda_0 = \Sigma_0^{-1}$
- **Noise precision:** $\beta = 1/\sigma^2$

The posterior is $P(\vec{w}|X, \vec{y}) = \mathcal{N}(\vec{w}; \vec{m}_N, S_N)$ where:

$$\boxed{S_N^{-1} = \Lambda_0 + \beta X^T X = \frac{1}{\tau^2}I + \frac{1}{\sigma^2}X^TX}$$

$$\boxed{\vec{m}_N = S_N\left(\Lambda_0 \vec{\mu}_0 + \beta X^T \vec{y}\right) = S_N \cdot \frac{1}{\sigma^2} X^T \vec{y} \quad \text{(with zero-mean prior)}}$$

### **Derivation (Completing the Square)**
The posterior is:
$$P(\vec{w}|X,\vec{y}) \propto P(\vec{y}|X,\vec{w}) P(\vec{w})$$
$$\propto \exp\left(-\frac{1}{2\sigma^2}\|\vec{y} - X\vec{w}\|^2\right) \exp\left(-\frac{1}{2\tau^2}\|\vec{w}\|^2\right)$$

Expanding and collecting $\vec{w}$ terms (completing the square):
$$\propto \exp\left(-\frac{1}{2}\left[\vec{w}^T \underbrace{\left(\frac{1}{\tau^2}I + \frac{1}{\sigma^2}X^TX\right)}_{S_N^{-1}} \vec{w} - 2\vec{w}^T \underbrace{\frac{1}{\sigma^2}X^T\vec{y}}_{S_N^{-1}\vec{m}_N}\right]\right)$$

This is the kernel of $\mathcal{N}(\vec{m}_N, S_N)$. ✅

### **Verify: Posterior Mean = MAP Estimate**
$$\vec{m}_N = S_N \frac{X^T\vec{y}}{\sigma^2} = \left(\frac{I}{\tau^2} + \frac{X^TX}{\sigma^2}\right)^{-1} \frac{X^T\vec{y}}{\sigma^2} = (X^TX + \frac{\sigma^2}{\tau^2}I)^{-1} X^T\vec{y} = \hat{\vec{w}}_{\text{Ridge}}$$

The posterior mean equals the MAP/Ridge estimate — as expected. But we now also have $S_N$ (posterior covariance), which tells us **how uncertain** we are about each weight.

---

## 4. Posterior Predictive Distribution

For a new test point $\vec{x}^*$, the prediction $y^* = \vec{x}^{*T}\vec{w} + \epsilon$ depends on the uncertain weights $\vec{w}$ and the noise $\epsilon$.

**Bayesian prediction:** Integrate (marginalize) over all possible weight values:
$$P(y^* | \vec{x}^*, X, \vec{y}) = \int P(y^* | \vec{x}^*, \vec{w}) P(\vec{w} | X, \vec{y}) \, d\vec{w}$$

Since both factors are Gaussian (linear-Gaussian model), this integral is tractable:

$$\boxed{P(y^* | \vec{x}^*, X, \vec{y}) = \mathcal{N}\left(y^*;\ \underbrace{\vec{m}_N^T \vec{x}^*}_{\text{predictive mean}},\ \underbrace{\sigma^2 + \vec{x}^{*T} S_N \vec{x}^*}_{\text{predictive variance}}\right)}$$

### **Understanding the Predictive Variance**
The variance has **two components**:

| Component | Source | When is it large? |
| :--- | :--- | :--- |
| $\sigma^2$ | **Observation noise** — irreducible; inherent randomness in data | Always (fixed) |
| $\vec{x}^{*T} S_N \vec{x}^*$ | **Parameter uncertainty** — how uncertain are we about $\vec{w}$? | Far from training data / small $N$ / few data in region |

**Key property:** The predictive variance is **automatically larger for extrapolation** (far from training data) and **smaller for interpolation** (near training data). No need to manually estimate confidence — it emerges from the Bayesian framework.

### **How $S_N$ Shrinks with Data**
$$S_N^{-1} = \frac{1}{\tau^2}I + \frac{1}{\sigma^2}X^TX$$

As $N \to \infty$: $X^TX \to \infty$, so $S_N \to 0$ (posterior becomes a point — MAP = MLE). The parameter uncertainty term $\vec{x}^{*T} S_N \vec{x}^* \to 0$, and predictions are driven entirely by model noise $\sigma^2$.

---

## 5. Log Marginal Likelihood (Evidence): Choosing Hyperparameters

How do we choose $\sigma^2$ and $\tau^2$ (or equivalently $\lambda = \sigma^2/\tau^2$)?

**Bayesian answer:** Maximize the **marginal likelihood** (type-II MLE / evidence approximation / empirical Bayes):
$$P(\vec{y} | X, \sigma^2, \tau^2) = \int P(\vec{y}|X,\vec{w}) P(\vec{w}|\tau^2) d\vec{w}$$

For Bayesian linear regression, this integral is tractable:
$$P(\vec{y} | X, \sigma^2, \tau^2) = \mathcal{N}\left(\vec{y}; \vec{0},\ \sigma^2 I + \tau^2 X X^T\right)$$

The **log marginal likelihood:**
$$\log P(\vec{y}|X) = -\frac{1}{2}\left[N\log(2\pi) + \log|C| + \vec{y}^T C^{-1} \vec{y}\right]$$
where $C = \sigma^2 I + \tau^2 X X^T$.

Maximize this over $\sigma^2$ and $\tau^2$ — no train/validation split needed! The model automatically selects hyperparameters based on how well it can explain the training data.

---

## 6. Interview Q&A

**Q1: What is Bayesian linear regression?**
> It treats the regression weights $\vec{w}$ as random variables with a prior distribution $P(\vec{w})$. After observing data, Bayes' theorem gives a posterior $P(\vec{w}|\mathcal{D})$ — a full distribution over all possible weight values. Predictions are made by integrating over this posterior, yielding a predictive distribution with uncertainty estimates.

**Q2: Show that Ridge regression is MAP estimation.**
> The log-posterior is $\log P(\vec{w}|X,\vec{y}) \propto -\frac{1}{2\sigma^2}\|\vec{y}-X\vec{w}\|^2 - \frac{1}{2\tau^2}\|\vec{w}\|^2$. Maximizing this is equivalent to minimizing $\|\vec{y}-X\vec{w}\|^2 + (\sigma^2/\tau^2)\|\vec{w}\|^2$, which is exactly Ridge regression with $\lambda = \sigma^2/\tau^2$. A Gaussian prior on weights with variance $\tau^2$ corresponds to L2 regularization with $\lambda = \sigma^2/\tau^2$.

**Q3: What is the closed-form posterior in Bayesian linear regression?**
> $P(\vec{w}|X,\vec{y}) = \mathcal{N}(\vec{m}_N, S_N)$ where $S_N^{-1} = \frac{1}{\tau^2}I + \frac{1}{\sigma^2}X^TX$ and $\vec{m}_N = \frac{1}{\sigma^2} S_N X^T\vec{y}$. The posterior mean equals the Ridge estimate.

**Q4: What is the posterior predictive distribution and what does its variance tell you?**
> For a new point $\vec{x}^*$: $p(y^*|\vec{x}^*, \mathcal{D}) = \mathcal{N}(y^*; \vec{m}_N^T \vec{x}^*, \sigma^2 + \vec{x}^{*T} S_N \vec{x}^*)$. The variance has two parts: $\sigma^2$ (irreducible noise) and $\vec{x}^{*T} S_N \vec{x}^*$ (parameter uncertainty). The second term is larger when $\vec{x}^*$ is far from the training data — predictions automatically become more uncertain in data-sparse regions.

**Q5: How does the posterior change as more data is observed?**
> The posterior precision $S_N^{-1} = \frac{1}{\tau^2}I + \frac{1}{\sigma^2}X^TX$ increases with each data point (more data = higher precision = tighter posterior). The posterior covariance $S_N$ decreases, concentrating the distribution around $\vec{m}_N$. As $N \to \infty$, the posterior concentrates at the MLE and the prior becomes irrelevant.

**Q6: What is empirical Bayes/Type-II MLE in this context?**
> Instead of choosing hyperparameters ($\sigma^2$, $\tau^2$) by cross-validation, maximize the marginal likelihood $P(\vec{y}|X,\sigma^2,\tau^2)$ (obtained by integrating out $\vec{w}$). This is a Bayesian approach to hyperparameter selection that avoids train/validation splits by using the full training data efficiently.

**Q7: Why can Bayesian linear regression detect extrapolation?**
> Because the predictive variance includes the term $\vec{x}^{*T} S_N \vec{x}^*$. When $\vec{x}^*$ is far from the training data, the covariance $S_N$ is large along that direction (few data points have constrained the weights in that direction). The inner product $\vec{x}^{*T} S_N \vec{x}^*$ becomes large, inflating the predictive variance. The model automatically signals higher uncertainty for extrapolation — something OLS/Ridge cannot do.

---

In [ ]:
# ─── CELL 1: Bayesian Linear Regression — From Scratch ───────────────────────
import numpy as np
import matplotlib.pyplot as plt

class BayesianLinearRegression:
    """
    Bayesian linear regression with Gaussian prior and Gaussian noise.
    Prior:     w ~ N(0, tau^2 * I)
    Likelihood: y | x, w ~ N(w^T x, sigma^2)
    Posterior:  w | X, y ~ N(m_N, S_N)
    """
    def __init__(self, sigma2=1.0, tau2=1.0):
        self.sigma2 = sigma2  # noise variance
        self.tau2 = tau2       # prior weight variance
        self.m_N = None
        self.S_N = None

    def fit(self, X, y):
        N, d = X.shape
        beta = 1 / self.sigma2   # noise precision
        alpha = 1 / self.tau2    # prior precision

        # Posterior precision: S_N^{-1} = alpha*I + beta*X^T X
        S_N_inv = alpha * np.eye(d) + beta * X.T @ X          # d x d
        self.S_N = np.linalg.inv(S_N_inv)                      # d x d

        # Posterior mean: m_N = S_N * beta * X^T y
        self.m_N = beta * self.S_N @ X.T @ y                   # d
        return self

    def predict(self, X_new):
        """Returns (predictive_mean, predictive_std) for each test point."""
        mean = X_new @ self.m_N                                             # N_new
        # Predictive variance: sigma^2 + x^T S_N x  (per test point)
        param_variance = np.einsum('nd,dd,nd->n', X_new, self.S_N, X_new)  # x^T S_N x
        total_variance = self.sigma2 + param_variance
        return mean, np.sqrt(total_variance)

    @property
    def map_weights(self):
        """MAP/Ridge estimate (= posterior mean)."""
        return self.m_N


# ── Experiment: 1D Bayesian regression ────────────────────────────────────────
np.random.seed(42)
true_w = np.array([1.5, -0.8])  # true weights [bias, slope]
sigma2_true = 0.3

def make_phi(x):
    """Feature map: [1, x] (add bias term)."""
    return np.column_stack([np.ones_like(x), x])

# Training data: deliberately sparse (few points) in a region
X_train = np.array([-2, -1.5, -1, 1.5, 2, 2.5])  # no points in (-1, 1.5)!
y_train = X_train * true_w[1] + true_w[0] + np.sqrt(sigma2_true) * np.random.randn(len(X_train))
Phi_train = make_phi(X_train)

# Test grid
x_test = np.linspace(-3, 4, 200)
Phi_test = make_phi(x_test)

# Fit Bayesian models with different tau^2 (prior strength)
tau2_values = [0.1, 1.0, 10.0, 100.0]
fig, axes = plt.subplots(2, 2, figsize=(14, 11))
axes = axes.flatten()

for ax, tau2 in zip(axes, tau2_values):
    blr = BayesianLinearRegression(sigma2=sigma2_true, tau2=tau2).fit(Phi_train, y_train)
    mu_pred, std_pred = blr.predict(Phi_test)

    # From OLS for comparison
    ols_w = np.linalg.lstsq(Phi_train, y_train, rcond=None)[0]
    ols_pred = Phi_test @ ols_w

    # True function
    y_true_line = x_test * true_w[1] + true_w[0]

    # Uncertainty bands
    ax.fill_between(x_test, mu_pred - 2*std_pred, mu_pred + 2*std_pred,
                    alpha=0.25, color='steelblue', label='±2σ predictive')
    ax.fill_between(x_test, mu_pred - std_pred, mu_pred + std_pred,
                    alpha=0.4, color='steelblue')
    ax.plot(x_test, mu_pred, 'b-', lw=2.5, label='Bayesian mean (MAP)')
    ax.plot(x_test, ols_pred, 'r--', lw=1.8, label='OLS (no uncertainty)', alpha=0.8)
    ax.plot(x_test, y_true_line, 'g-', lw=1.5, alpha=0.7, label='True function')
    ax.scatter(X_train, y_train, zorder=5, s=60, color='black', label='Training data')

    # Highlight the gap region (no training data)
    ax.axvspan(-1, 1.5, alpha=0.05, color='red', label='No training data here')

    lam = sigma2_true / tau2
    ax.set_title(f'τ²={tau2} (λ=σ²/τ²={lam:.2f} — Ridge λ equivalent)\n'
                 f'Large τ²=weak prior; small τ²=strong prior', fontsize=9)
    ax.set_xlim(-3, 4); ax.set_ylim(-5, 6)
    ax.grid(alpha=0.3); ax.legend(fontsize=7)

plt.suptitle('Bayesian Linear Regression: Predictive Uncertainty across Prior Strengths\n'
             'Notice: uncertainty is LARGER in the gap (no data) — automatic extrapolation detection!',
             fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# ─── CELL 2: Sequential Bayesian Updating — Posterior Concentrates with Data ───
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse
import matplotlib.transforms as transforms

np.random.seed(7)
true_w = np.array([0.5, -1.5])  # [bias, slope]
sigma2 = 0.5
tau2 = 5.0

# Generate data
X_all = np.random.uniform(-3, 3, 100)
y_all = X_all * true_w[1] + true_w[0] + np.random.randn(100) * np.sqrt(sigma2)
Phi_all = np.column_stack([np.ones_like(X_all), X_all])

# Show at different N
N_list = [1, 3, 10, 30, 100]
x_test = np.linspace(-4, 4, 200)
Phi_test = np.column_stack([np.ones_like(x_test), x_test])

fig, axes = plt.subplots(2, len(N_list), figsize=(20, 9))

def eigplot_cov(ax, mean, cov, n_std=2, color='steelblue'):
    """Draw a 2D covariance ellipse."""
    vals, vecs = np.linalg.eigh(cov)
    angle = np.degrees(np.arctan2(*vecs[::-1, 0]))
    width, height = 2 * n_std * np.sqrt(np.abs(vals))
    ell = Ellipse(xy=mean, width=width, height=height, angle=angle,
                  edgecolor=color, fc=color, alpha=0.3, lw=2)
    ax.add_patch(ell)
    ax.plot(*mean, '+', color=color, ms=15, mew=2.5)

for col, N in enumerate(N_list):
    Phi_N = Phi_all[:N]; y_N = y_all[:N]
    blr = BayesianLinearRegression(sigma2=sigma2, tau2=tau2).fit(Phi_N, y_N)
    mu_pred, std_pred = blr.predict(Phi_test)

    # Top row: posterior over weights in 2D
    ax_top = axes[0][col]
    w0 = np.linspace(-2, 3, 100)
    w1 = np.linspace(-4, 1, 100)
    W0, W1 = np.meshgrid(w0, w1)
    W_grid = np.stack([W0.ravel(), W1.ravel()], axis=1)  # (10000, 2)
    from scipy import stats as sc_stats
    rv = sc_stats.multivariate_normal(mean=blr.m_N, cov=blr.S_N)
    Z = rv.pdf(W_grid).reshape(W0.shape)
    ax_top.contourf(W0, W1, Z, levels=15, cmap='Blues')
    ax_top.plot(*true_w, 'r*', ms=15, zorder=10, label='True w')
    ax_top.plot(*blr.m_N, 'bX', ms=10, zorder=10, label='Post. mean')
    ax_top.set_xlabel('w₀ (bias)'); ax_top.set_ylabel('w₁ (slope)')
    ax_top.set_title(f'N={N}: Posterior P(w|X,y)',  fontsize=10)
    if col == 0: ax_top.legend(fontsize=8, loc='lower right')

    # Bottom row: predictive distribution
    ax_bot = axes[1][col]
    ax_bot.fill_between(x_test, mu_pred - 2*std_pred, mu_pred + 2*std_pred,
                        alpha=0.25, color='steelblue')
    ax_bot.fill_between(x_test, mu_pred - std_pred, mu_pred + std_pred,
                         alpha=0.4, color='steelblue')
    ax_bot.plot(x_test, mu_pred, 'b-', lw=2)
    ax_bot.plot(x_test, x_test*true_w[1]+true_w[0], 'g--', lw=1.5, alpha=0.7, label='True fn')
    ax_bot.scatter(X_all[:N], y_all[:N], s=20, c='black', zorder=5)
    ax_bot.set_title(f'N={N}: Predictive P(y*|x*,data)', fontsize=10)
    ax_bot.set_xlim(-4, 4); ax_bot.set_ylim(-8, 8)
    ax_bot.grid(alpha=0.3)

plt.suptitle('Sequential Bayesian Updating: Posterior Gets Tighter and Predictions More Confident with More Data',
             fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

# Show Ridge connection numerically
print("=== MAP (Posterior Mean) vs Ridge ===\n")
for N in [5, 20, 50]:
    Phi_N, y_N = Phi_all[:N], y_all[:N]
    blr_fit = BayesianLinearRegression(sigma2=sigma2, tau2=tau2).fit(Phi_N, y_N)
    # Ridge with equivalent lambda = sigma2 / tau2
    lam = sigma2 / tau2
    w_ridge = np.linalg.solve(Phi_N.T @ Phi_N + lam * np.eye(2), Phi_N.T @ y_N)
    print(f"N={N:3d}: MAP={blr_fit.m_N.round(3)}, Ridge={w_ridge.round(3)}, Match={np.allclose(blr_fit.m_N, w_ridge)}")